# Comparing the Pleiades candidates with a published catalogue

This notebook compares our provisional candidates with an independent published membership list. Agreement can estimate how many established members we recover, while disagreements help reveal possible contaminants, missed members, and differences in the scientific definitions used by two analyses.

A catalogue match is not absolute ground truth. Published memberships also depend on their input data, search region, clustering method, probability threshold, and treatment of extended cluster structure.

## 1. Choose the reference catalogue

We use the Pleiades member table from [Hunt & Reffert (2023), *Improving the open cluster census II: An all-sky cluster catalogue with Gaia DR3*](https://doi.org/10.1051/0004-6361/202346285). The full catalogue and cluster-member tables are published through CDS as catalogue [`J/A+A/673/A114`](https://cdsarc.cds.unistra.fr/viz-bin/cat/J/A+A/673/A114).

This is a useful comparison because:

- it is based on Gaia DR3, like our project,
- it provides Gaia DR3 source identifiers for individual members,
- it resulted from a homogeneous all-sky cluster search rather than our targeted Pleiades analysis,
- it includes a membership probability that we can examine rather than hiding catalogue uncertainty.

The catalogue calls the Pleiades `Melotte_22`. Its method and selection function differ from ours, so missing or additional sources require investigation rather than automatic rejection.

## 2. Imports and project paths

Astroquery's `Vizier` interface retrieves the published CDS table. We cache the unmodified response in `data/raw/` so subsequent runs remain reproducible and do not depend unnecessarily on network access.

In [ ]:
import sys
from pathlib import Path

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
from astropy.coordinates import SkyCoord
from astropy.table import MaskedColumn, Table
from astroquery.vizier import Vizier

print(f"Python interpreter: {sys.executable}")
print(f"Python version: {sys.version.split()[0]}")

In [ ]:
project_root = Path.cwd()

if project_root.name.lower() == "notebooks":
    project_root = project_root.parent

candidate_data_path = (
    project_root
    / "data"
    / "processed"
    / "gaia_dr3_pleiades_candidates_cmd_validated.fits"
)
published_catalogue_path = (
    project_root
    / "data"
    / "raw"
    / "hunt_reffert_2023_melotte_22_members.fits"
)
processed_data_dir = project_root / "data" / "processed"
figures_dir = project_root / "figures"

processed_data_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)
published_catalogue_path.parent.mkdir(parents=True, exist_ok=True)

print(f"Our candidate catalogue exists: {candidate_data_path.exists()}")
print(f"Published catalogue cached locally: {published_catalogue_path.exists()}")

## 3. Load our CMD-validated candidates

We retain all 1,185 astrometric candidates for the catalogue comparison. The CMD availability and photometric-consistency columns are supporting diagnostics; they are not used to remove sources before matching.

In [ ]:
if not candidate_data_path.exists():
    raise FileNotFoundError(
        "The CMD-validated candidate catalogue is missing. Run Notebook 04 first."
    )

our_candidates = Table.read(candidate_data_path, format="fits")

assert len(np.unique(our_candidates["source_id"])) == len(our_candidates)
print(f"Our candidates: {len(our_candidates):,}")
print(f"Our columns: {len(our_candidates.colnames)}")

## 4. Download or load the published Pleiades members

If the raw reference file already exists, we load it directly. Otherwise, we query only the `members` table and constrain its `Name` column to `Melotte_22`. `row_limit = -1` requests every matching member rather than VizieR's default limited preview.

The downloaded table is written without filtering or column changes. This preserves the published result as raw external data.

In [ ]:
if published_catalogue_path.exists():
    published_members = Table.read(
        published_catalogue_path, format="fits"
    )
    catalogue_source = "local cached file"
else:
    vizier = Vizier(columns=["*"], row_limit=-1)
    query_result = vizier.query_constraints(
        catalog="J/A+A/673/A114/members",
        Name="Melotte_22",
    )

    if len(query_result) != 1 or len(query_result[0]) == 0:
        raise RuntimeError(
            "The CDS query did not return the expected Pleiades member table."
        )

    published_members = query_result[0]
    published_members.write(
        published_catalogue_path, format="fits", overwrite=False
    )
    catalogue_source = "new CDS/VizieR query"

print(f"Loaded from: {catalogue_source}")
print(f"Published Pleiades members: {len(published_members):,}")

## 5. Validate the published table structure

The essential matching column is `GaiaDR3`. We also retain the published membership probability, sky coordinates, astrometry, and photometry so disagreements can later be investigated scientifically.

We check that source identifiers are unique within the Pleiades table before performing a one-to-one identifier comparison.

In [ ]:
published_required_columns = [
    "Name",
    "GaiaDR3",
    "Prob",
    "RA_ICRS",
    "DE_ICRS",
    "pmRA",
    "pmDE",
    "Plx",
]
published_missing_columns = [
    name
    for name in published_required_columns
    if name not in published_members.colnames
]

if published_missing_columns:
    raise ValueError(
        f"Published table is missing columns: {published_missing_columns}"
    )

published_source_ids = np.asarray(published_members["GaiaDR3"], dtype=np.int64)
assert len(np.unique(published_source_ids)) == len(published_source_ids)
published_cluster_names = np.asarray(
    published_members["Name"]
).astype(str)
assert np.all(published_cluster_names == "Melotte_22")

print("Published table validation passed.")
print(f"Unique Gaia DR3 identifiers: {len(np.unique(published_source_ids)):,}")
published_members[:5]

### Comparison checkpoint

We now have two validated Gaia DR3 identifier lists: our 1,185 candidates and 1,721 published members. The next section will calculate their intersection and distinguish three populations: recovered published members, our additional candidates, and published members missed by our selection.

## 6. Match the Gaia DR3 source identifiers

Because both catalogues use Gaia DR3 identifiers, matching is an exact set-membership operation rather than a positional nearest-neighbour search. `np.isin` returns a Boolean array showing whether each identifier appears in the other catalogue.

We create three scientifically useful populations:

1. **Shared sources:** selected by both analyses.
2. **Our additional candidates:** selected by our DBSCAN analysis but absent from the published Pleiades list.
3. **Published-only sources:** present in Hunt & Reffert but absent from our candidate list.

These labels describe catalogue agreement, not true-positive or false-positive status.

In [ ]:
our_source_ids = np.asarray(our_candidates["source_id"], dtype=np.int64)

our_is_published = np.isin(our_source_ids, published_source_ids)
published_is_ours = np.isin(published_source_ids, our_source_ids)

shared_candidates = our_candidates[our_is_published].copy()
our_additional_candidates = our_candidates[~our_is_published].copy()
published_only_members = published_members[~published_is_ours].copy()

overlap_summary = Table(
    rows=[
        ("Our candidates", len(our_candidates)),
        ("Published members", len(published_members)),
        ("Shared by both", len(shared_candidates)),
        ("Our additional candidates", len(our_additional_candidates)),
        ("Published only", len(published_only_members)),
    ],
    names=("population", "sources"),
)
overlap_summary

## 7. Account for our 3-degree sky footprint

The initial Gaia query covered a circle of radius 3 degrees around RA = 56.75 deg and Dec = 24.12 deg. Hunt & Reffert's published member list extends farther into the Pleiades surroundings. A published source outside our input footprint was never available to DBSCAN, so counting it as a missed detection would be unfair.

We calculate each published member's angular separation from our query centre and divide the published list into sources inside and outside the original footprint. This is a geometric comparison, not another membership cut.

In [ ]:
query_centre = SkyCoord(ra=56.75 * u.deg, dec=24.12 * u.deg)
published_coordinates = SkyCoord(
    ra=np.asarray(published_members["RA_ICRS"], dtype=float) * u.deg,
    dec=np.asarray(published_members["DE_ICRS"], dtype=float) * u.deg,
)
published_separation = published_coordinates.separation(query_centre)
published_inside_footprint = published_separation <= 3 * u.deg

published_recovered_inside = published_inside_footprint & published_is_ours
published_missed_inside = published_inside_footprint & ~published_is_ours
published_outside_footprint = ~published_inside_footprint

footprint_summary = Table(
    rows=[
        ("Published members inside 3 deg", np.count_nonzero(published_inside_footprint)),
        ("Recovered inside 3 deg", np.count_nonzero(published_recovered_inside)),
        ("Missed inside 3 deg", np.count_nonzero(published_missed_inside)),
        ("Published members outside 3 deg", np.count_nonzero(published_outside_footprint)),
    ],
    names=("population", "sources"),
)
footprint_summary

In [ ]:
published_agreement_fraction = len(shared_candidates) / len(our_candidates)
footprint_recovery_fraction = (
    np.count_nonzero(published_recovered_inside)
    / np.count_nonzero(published_inside_footprint)
)

print(
    "Fraction of our candidates in the published catalogue: "
    f"{published_agreement_fraction:.2%}"
)
print(
    "Fraction of published members recovered inside our footprint: "
    f"{footprint_recovery_fraction:.2%}"
)

### Interpret the agreement carefully

The first fraction asks, 'How many of our candidates also appear in this publication?' The second asks, 'How many published members that were observable within our query footprint did our selection recover?'

They are not formal precision and recall measurements because the published catalogue is not perfect ground truth. Differences may reflect our RUWE cut, the local astrometric window, DBSCAN density boundaries, published membership probabilities, or genuine methodological disagreements.

## 8. Compare agreement for DBSCAN core and border points

Core and border labels quantify local density support in our DBSCAN run. Comparing their catalogue agreement tests whether the lower-density border population is less consistently selected by the independent analysis. We also count how many of our additional candidates pass the CMD photometric-consistency check.

In [ ]:
our_is_core = np.asarray(our_candidates["dbscan_core"], dtype=bool)

role_agreement = Table(
    names=("dbscan_role", "total", "published_match", "not_published", "match_percent"),
    dtype=("U10", int, int, int, float),
)

for role_name, role_mask in [("core", our_is_core), ("border", ~our_is_core)]:
    total = np.count_nonzero(role_mask)
    matched = np.count_nonzero(role_mask & our_is_published)
    role_agreement.add_row(
        (role_name, total, matched, total - matched, 100 * matched / total)
    )

role_agreement["match_percent"].format = ".1f"
print(
    "Photometrically consistent among our additional candidates: "
    f"{np.count_nonzero(our_additional_candidates['photometry_consistent_3sigma']):,}"
)
role_agreement

### Overlap checkpoint

The high shared fraction strongly supports our DBSCAN selection. The 176 published members missed inside the footprint and our 31 additional candidates are the scientifically interesting disagreement sets. Next, we will compare their membership probabilities, astrometry, sky positions, and CMD quality to understand why the methods differ.

## 9. Trace missed published members through our pipeline

The 176 published members missed inside our footprint can disappear at several stages:

1. absence from the raw Gaia query,
2. missing core astrometry,
3. failure of the RUWE or visibility-period criteria,
4. exclusion by the broad local astrometric window,
5. classification as another cluster or noise by baseline DBSCAN.

We load the raw and cleaned catalogues and use Gaia source identifiers to audit these stages. This turns a vague disagreement into a reproducible account of which choice caused each omission.

In [ ]:
raw_data_path = (
    project_root / "data" / "raw" / "gaia_dr3_pleiades_3deg_all_sources.fits"
)
cleaned_data_path = (
    project_root / "data" / "processed" / "gaia_dr3_pleiades_3deg_cleaned.fits"
)

gaia_raw = Table.read(raw_data_path, format="fits")
gaia_clean = Table.read(cleaned_data_path, format="fits")

raw_source_ids = np.asarray(gaia_raw["source_id"], dtype=np.int64)
clean_source_ids = np.asarray(gaia_clean["source_id"], dtype=np.int64)
missed_published_ids = published_source_ids[published_missed_inside]

print(f"Published members missed inside footprint: {len(missed_published_ids):,}")

In [ ]:
missed_is_raw = np.isin(missed_published_ids, raw_source_ids)
missed_is_clean = np.isin(missed_published_ids, clean_source_ids)

clean_rows_for_missed = gaia_clean[
    np.isin(clean_source_ids, missed_published_ids)
]
missed_clean_inside_window = (
    (clean_rows_for_missed["parallax"] >= 2.0)
    & (clean_rows_for_missed["parallax"] <= 12.0)
    & (clean_rows_for_missed["pmra"] >= 0.0)
    & (clean_rows_for_missed["pmra"] <= 40.0)
    & (clean_rows_for_missed["pmdec"] >= -70.0)
    & (clean_rows_for_missed["pmdec"] <= -25.0)
)

n_not_in_raw = np.count_nonzero(~missed_is_raw)
n_removed_by_cleaning = np.count_nonzero(missed_is_raw & ~missed_is_clean)
n_outside_analysis_window = len(clean_rows_for_missed) - np.count_nonzero(
    missed_clean_inside_window
)
n_not_selected_by_dbscan = np.count_nonzero(missed_clean_inside_window)

missed_stage_summary = Table(
    rows=[
        ("Absent from raw query", n_not_in_raw),
        ("Removed during cleaning", n_removed_by_cleaning),
        ("Outside local astrometric window", n_outside_analysis_window),
        ("Inside window but not selected by DBSCAN", n_not_selected_by_dbscan),
    ],
    names=("reason", "published_members"),
)
missed_stage_summary

### Identify the cleaning criterion responsible

Notebook 02 removed sources for missing core astrometry or for failing the baseline RUWE/visibility criteria. We inspect the corresponding raw rows to determine which condition accounts for the 154 cleaning-stage omissions.

In [ ]:
cleaning_omission_ids = missed_published_ids[missed_is_raw & ~missed_is_clean]
cleaning_omission_rows = gaia_raw[
    np.isin(raw_source_ids, cleaning_omission_ids)
]

has_finite_core_astrometry = np.ones(
    len(cleaning_omission_rows), dtype=bool
)
for name in ["parallax", "pmra", "pmdec"]:
    column = cleaning_omission_rows[name]
    has_finite_core_astrometry &= (
        ~np.ma.getmaskarray(column)
        & np.isfinite(np.ma.filled(column, np.nan))
    )

fails_ruwe = (
    has_finite_core_astrometry
    & (np.asarray(cleaning_omission_rows["ruwe"]) >= 1.4)
)
fails_visibility = (
    has_finite_core_astrometry
    & ~fails_ruwe
    & (np.asarray(cleaning_omission_rows["visibility_periods_used"]) < 9)
)

cleaning_reason_summary = Table(
    rows=[
        ("Missing core astrometry", np.count_nonzero(~has_finite_core_astrometry)),
        ("RUWE >= 1.4", np.count_nonzero(fails_ruwe)),
        ("Visibility periods < 9", np.count_nonzero(fails_visibility)),
    ],
    names=("cleaning_reason", "published_members"),
)
cleaning_reason_summary

## 10. Compare published membership probabilities

Hunt & Reffert provide a membership probability for each source. We compare the distributions for recovered members, members omitted by RUWE, and the 22 sources retained through cleaning but not selected by DBSCAN.

This probability belongs to the published model and is not directly calibrated against our DBSCAN labels. Nevertheless, it helps show whether our disagreements preferentially involve lower-confidence published members.

In [ ]:
published_cleaned = np.isin(published_source_ids, clean_source_ids)
probability_groups = {
    "Recovered": published_recovered_inside,
    "Removed by RUWE": published_missed_inside & ~published_cleaned,
    "Not selected by DBSCAN": published_missed_inside & published_cleaned,
}

probability_summary = Table(
    names=("group", "sources", "median_probability", "mean_probability"),
    dtype=("U30", int, float, float),
)

for group_name, group_mask in probability_groups.items():
    probabilities = np.asarray(published_members["Prob"][group_mask], dtype=float)
    probability_summary.add_row(
        (group_name, len(probabilities), np.median(probabilities), np.mean(probabilities))
    )

for name in probability_summary.colnames[2:]:
    probability_summary[name].format = ".3f"
probability_summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
probability_bins = np.linspace(0, 1, 21)

for group_name, group_mask in probability_groups.items():
    ax.hist(
        published_members["Prob"][group_mask],
        bins=probability_bins,
        histtype="step",
        linewidth=2,
        label=f"{group_name} ({np.count_nonzero(group_mask):,})",
    )

ax.set(
    xlabel="Published membership probability",
    ylabel="Number of sources",
    title="Published probabilities by our pipeline outcome",
)
ax.legend()
ax.grid(alpha=0.2)

probability_figure_path = figures_dir / "05_published_probability_comparison.png"
fig.savefig(probability_figure_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved figure to: {probability_figure_path}")

### Missed-member conclusion

The dominant source of disagreement is the RUWE < 1.4 quality choice, not missing data, query coverage, or the broad astrometric window. This is scientifically plausible because genuine cluster binaries can have elevated RUWE. The result reinforces why Notebook 02 described the threshold as an exploratory quality decision rather than a universal membership rule.

The 22 cleaned sources not selected by DBSCAN have much lower published membership probabilities on average. This suggests that the density algorithm mainly misses lower-confidence peripheral cases, while the RUWE cut removes a mixture that includes some stronger published candidates. A useful future sensitivity test would rerun membership without the RUWE cut and inspect those recovered high-RUWE sources separately.

## 11. Inspect our additional candidates

Our 31 additional candidates are absent from the Hunt & Reffert member table, but absence alone does not show whether they are contaminants or plausible new members. We examine three kinds of supporting evidence:

- whether they are DBSCAN core or border points,
- whether they have the photometry required for a CMD,
- whether that photometry passes the corrected flux-excess criterion.

We then locate them relative to the shared population in the same astrometric projections used earlier and on the CMD.

In [ ]:
additional_is_core = np.asarray(
    our_additional_candidates["dbscan_core"], dtype=bool
)
additional_has_cmd = np.asarray(
    our_additional_candidates["cmd_available"], dtype=bool
)
additional_photometry_consistent = np.asarray(
    our_additional_candidates["photometry_consistent_3sigma"],
    dtype=bool,
)

additional_summary = Table(
    rows=[
        ("All additional candidates", len(our_additional_candidates)),
        ("DBSCAN core", np.count_nonzero(additional_is_core)),
        ("DBSCAN border", np.count_nonzero(~additional_is_core)),
        ("CMD available", np.count_nonzero(additional_has_cmd)),
        ("CMD unavailable", np.count_nonzero(~additional_has_cmd)),
        ("Photometrically consistent", np.count_nonzero(additional_photometry_consistent)),
    ],
    names=("property", "sources"),
)
additional_summary

### Locate the additional candidates

The shared catalogue population is shown in light grey. Additional candidates are divided into:

- red crosses: CMD photometry unavailable,
- orange open circles: CMD available but outside the 3-sigma flux-excess envelope,
- green stars: CMD available and photometrically consistent.

Only sources with CMD measurements can appear in the third panel.

In [ ]:
additional_no_cmd = ~additional_has_cmd
additional_bad_photometry = additional_has_cmd & ~additional_photometry_consistent
additional_good_photometry = additional_photometry_consistent

additional_groups = [
    (additional_no_cmd, "CMD unavailable", "tab:red", "x"),
    (additional_bad_photometry, "Inconsistent photometry", "tab:orange", "o"),
    (additional_good_photometry, "Consistent photometry", "tab:green", "*"),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)

axes[0].scatter(
    shared_candidates["pmra"], shared_candidates["pmdec"],
    s=8, color="lightgrey", alpha=0.6, label="Shared"
)
axes[1].scatter(
    shared_candidates["parallax"], shared_candidates["pmdec"],
    s=8, color="lightgrey", alpha=0.6
)
axes[2].scatter(
    shared_candidates["bp_rp_colour"], shared_candidates["absolute_g_mag"],
    s=8, color="lightgrey", alpha=0.6
)

for group_mask, label, colour_name, marker in additional_groups:
    plot_kwargs = dict(
        s=55, color=colour_name, marker=marker, linewidths=1.3, label=label
    )
    axes[0].scatter(
        our_additional_candidates["pmra"][group_mask],
        our_additional_candidates["pmdec"][group_mask],
        **plot_kwargs,
    )
    axes[1].scatter(
        our_additional_candidates["parallax"][group_mask],
        our_additional_candidates["pmdec"][group_mask],
        **plot_kwargs,
    )

    cmd_group_mask = group_mask & additional_has_cmd
    axes[2].scatter(
        our_additional_candidates["bp_rp_colour"][cmd_group_mask],
        our_additional_candidates["absolute_g_mag"][cmd_group_mask],
        **plot_kwargs,
    )

axes[0].set(
    xlabel=r"$\mu_{\alpha*}$ (mas yr$^{-1}$)",
    ylabel=r"$\mu_{\delta}$ (mas yr$^{-1}$)",
    title="Proper-motion plane",
)
axes[1].set(
    xlabel="Parallax (mas)",
    ylabel=r"$\mu_{\delta}$ (mas yr$^{-1}$)",
    title="Parallax projection",
)
axes[2].set(
    xlabel=r"$G_{BP}-G_{RP}$ (mag)",
    ylabel=r"$M_G$ (mag)",
    title="Colour-magnitude diagram",
)
axes[2].invert_yaxis()
axes[0].legend(markerscale=1.2, fontsize=8)

additional_figure_path = figures_dir / "05_additional_candidate_diagnostics.png"
fig.savefig(additional_figure_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved figure to: {additional_figure_path}")

### Additional-candidate interpretation

The additional candidates are astrometrically connected to the DBSCAN cluster, and most are core rather than border points. However, more than half lack complete CMD photometry and most of the remainder fail the BP/RP consistency diagnostic. This likely helps explain why an independent pipeline with its own photometric validation did not retain them.

The four photometrically consistent additional candidates deserve individual follow-up as plausible catalogue additions. The others should remain flagged rather than automatically rejected: missing or inconsistent Gaia photometry can arise from faintness, crowding, variability, or unresolved multiplicity.

## 12. Visualize the missed published members

The published catalogue provides astrometry and sky coordinates for every member, including sources removed from our processed table. We compare:

- recovered members inside our footprint,
- members omitted because RUWE was at least 1.4,
- members that survived cleaning but were not selected by DBSCAN.

If the DBSCAN omissions lie near the edge of the cluster in astrometric space, their lower published probabilities and our density-based rejection have a consistent explanation. If high-RUWE omissions occupy the main cluster locus, their absence reflects the quality policy rather than incompatible astrometry.

In [ ]:
recovered_mask = probability_groups["Recovered"]
ruwe_omission_mask = probability_groups["Removed by RUWE"]
dbscan_omission_mask = probability_groups["Not selected by DBSCAN"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)

axes[0].scatter(
    published_members["pmRA"][recovered_mask],
    published_members["pmDE"][recovered_mask],
    s=8, color="lightgrey", alpha=0.6, label="Recovered",
)
axes[1].scatter(
    published_members["Plx"][recovered_mask],
    published_members["pmDE"][recovered_mask],
    s=8, color="lightgrey", alpha=0.6,
)
axes[2].scatter(
    published_members["RA_ICRS"][recovered_mask],
    published_members["DE_ICRS"][recovered_mask],
    s=8, color="lightgrey", alpha=0.6,
)

missed_groups = [
    (ruwe_omission_mask, "RUWE omission", "tab:orange", "o"),
    (dbscan_omission_mask, "DBSCAN omission", "tab:red", "x"),
]

for group_mask, label, colour_name, marker in missed_groups:
    axes[0].scatter(
        published_members["pmRA"][group_mask],
        published_members["pmDE"][group_mask],
        s=28, color=colour_name, marker=marker, alpha=0.8, label=label,
    )
    axes[1].scatter(
        published_members["Plx"][group_mask],
        published_members["pmDE"][group_mask],
        s=28, color=colour_name, marker=marker, alpha=0.8,
    )
    axes[2].scatter(
        published_members["RA_ICRS"][group_mask],
        published_members["DE_ICRS"][group_mask],
        s=28, color=colour_name, marker=marker, alpha=0.8,
    )

axes[0].set(
    xlabel=r"$\mu_{\alpha*}$ (mas yr$^{-1}$)",
    ylabel=r"$\mu_{\delta}$ (mas yr$^{-1}$)",
    title="Proper-motion plane",
)
axes[1].set(
    xlabel="Parallax (mas)",
    ylabel=r"$\mu_{\delta}$ (mas yr$^{-1}$)",
    title="Parallax projection",
)
axes[2].set(
    xlabel="Right ascension (deg)",
    ylabel="Declination (deg)",
    title="Sky distribution inside 3 deg",
)
axes[2].invert_xaxis()
axes[0].legend(markerscale=1.4, fontsize=8)

missed_figure_path = figures_dir / "05_missed_published_member_diagnostics.png"
fig.savefig(missed_figure_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved figure to: {missed_figure_path}")

### Interpretation

The high-RUWE sources largely overlap the recovered cluster locus, so their omission should be understood as a deliberate reliability/completeness trade-off. Their astrometry may be less reliable under Gaia's single-source model, but many can still be genuine binaries or otherwise complex Pleiades systems.

The DBSCAN omissions are fewer and tend to occupy lower-density edges of the astrometric structure. Combined with their low published probabilities, this suggests they are ambiguous peripheral cases rather than evidence that DBSCAN missed a second dense Pleiades population. Sky position alone does not resolve their status because members and field contaminants can both lie inside the same projected region.

## 13. Save the catalogue-comparison results

We preserve all 1,185 of our candidates and add three comparison columns:

- `published_match`: whether the Gaia DR3 source appears in Hunt & Reffert's Pleiades list,
- `published_probability`: their membership probability, masked when no published match exists,
- `comparison_status`: either `shared` or `our_additional`.

We also save a separate audit table for the 176 published members missed inside our footprint. Its `omission_reason` records whether the source was removed by RUWE or survived cleaning but was not selected by DBSCAN. These are derived products; neither original input catalogue is overwritten.

In [ ]:
compared_candidates = our_candidates.copy()
compared_candidates["published_match"] = our_is_published
compared_candidates["comparison_status"] = np.where(
    our_is_published, "shared", "our_additional"
)

published_probability_lookup = {
    int(source_id): float(probability)
    for source_id, probability in zip(
        published_source_ids, published_members["Prob"]
    )
}
published_probability_values = np.full(len(compared_candidates), np.nan)
for index, source_id in enumerate(our_source_ids):
    if int(source_id) in published_probability_lookup:
        published_probability_values[index] = published_probability_lookup[
            int(source_id)
        ]

compared_candidates["published_probability"] = MaskedColumn(
    data=published_probability_values,
    mask=~our_is_published,
)
compared_candidates.meta["REFCAT"] = "J/A+A/673/A114"
compared_candidates.meta["REFCLUST"] = "Melotte_22"

print(f"Rows preserved: {len(compared_candidates):,}")
print(f"Shared sources: {np.count_nonzero(compared_candidates['published_match']):,}")
print(f"Our additional sources: {np.count_nonzero(~compared_candidates['published_match']):,}")

In [ ]:
published_missed_audit = published_members[published_missed_inside].copy()
audit_source_ids = np.asarray(
    published_missed_audit["GaiaDR3"], dtype=np.int64
)
audit_survived_cleaning = np.isin(audit_source_ids, clean_source_ids)
published_missed_audit["omission_reason"] = np.where(
    audit_survived_cleaning,
    "not_selected_by_dbscan",
    "ruwe_greater_equal_1.4",
)
published_missed_audit["separation_from_query_centre_deg"] = (
    published_separation[published_missed_inside].to_value(u.deg)
)
published_missed_audit["separation_from_query_centre_deg"].unit = u.deg

print(f"Missed-member audit rows: {len(published_missed_audit):,}")
print(
    "RUWE omissions: "
    f"{np.count_nonzero(~audit_survived_cleaning):,}"
)
print(
    "DBSCAN omissions: "
    f"{np.count_nonzero(audit_survived_cleaning):,}"
)

In [ ]:
compared_candidates_path = (
    processed_data_dir
    / "gaia_dr3_pleiades_candidates_catalogue_compared.fits"
)
missed_audit_path = (
    processed_data_dir
    / "hunt_reffert_2023_members_missed_inside_3deg.fits"
)

compared_candidates.write(
    compared_candidates_path, format="fits", overwrite=True
)
published_missed_audit.write(
    missed_audit_path, format="fits", overwrite=True
)

saved_compared_candidates = Table.read(
    compared_candidates_path, format="fits"
)
saved_missed_audit = Table.read(missed_audit_path, format="fits")

assert len(saved_compared_candidates) == 1_185
assert np.count_nonzero(saved_compared_candidates["published_match"]) == 1_154
assert np.count_nonzero(
    np.ma.getmaskarray(saved_compared_candidates["published_probability"])
) == 31
assert saved_compared_candidates.meta["REFCAT"] == "J/A+A/673/A114"
assert len(saved_missed_audit) == 176
assert np.count_nonzero(
    np.asarray(saved_missed_audit["omission_reason"]).astype(str)
    == "ruwe_greater_equal_1.4"
) == 154
assert np.count_nonzero(
    np.asarray(saved_missed_audit["omission_reason"]).astype(str)
    == "not_selected_by_dbscan"
) == 22

print("Both comparison files passed validation.")
print(f"Candidate comparison: {compared_candidates_path}")
print(f"Missed-member audit: {missed_audit_path}")

## 14. Final conclusions and limitations

Our targeted Gaia DR3 analysis identifies 1,185 provisional Pleiades candidates. Of these, 1,154 (97.38%) also appear in Hunt & Reffert (2023). Inside our original 3-degree footprint, we recover 86.77% of their published members. The agreement is especially strong for DBSCAN core points.

The comparison clarifies the disagreements:

- 391 published members lie outside our query footprint and were never available to our pipeline.
- 154 in-footprint members were removed only by RUWE >= 1.4, illustrating a reliability-versus-completeness trade-off and the risk of excluding genuine binaries.
- 22 cleaned published members were not selected by DBSCAN; they are mostly lower-probability, low-density peripheral cases.
- 31 of our candidates are absent from the publication. Most lack reliable CMD validation, while four have consistent photometry and merit individual follow-up.

The astrometric overdensity, stable DBSCAN behaviour, coherent CMD sequence, spatial concentration, and high published-catalogue overlap jointly support the candidate selection. However, this remains a candidate catalogue rather than a definitive probabilistic membership model. A stronger follow-up analysis could model Gaia uncertainties and correlations, treat high-RUWE sources separately, correct parallax and extinction systematics, expand the sky footprint to the tidal tails, and compare additional membership catalogues.